### Manual: 16-notes

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


## Reading the result

Artifacts land in `/kaggle/working/smoke_test/artifacts` (small JSON, safe to
download with `kaggle kernels output`) and logs in `.../logs`.

| artifact | what it proves |
|---|---|
| `system.json` | GPU, driver, which CUDA toolkits exist, dataset mount |
| `venv.json`, `pinned_wheels.json`, `env.json` | the pinned stack is really Python 3.8 + torch 1.10.1+cu113 + mmcv-full 1.4.0 + mmdet 2.20.0 |
| `pin.json` | clone is at `da93b27`, tree clean, no `add_depth_features` / `height_expand` |
| `cuda_toolchain.json` | which `nvcc` compiled the ops, and whether torch's major-version guard was bypassed |
| `build_ops.json`, `ops.json` | every extension imports **and** `bev_pool` + hard voxelization ran on the T4 |
| `checkpoint_file.json` | URL that actually served the checkpoint, size, sha256, torch-archive magic |
| `config_snapshot.json`, `cfg_resolved.json`, `cfg_checks.json` | exact config chain + sha256, resolved values, invariants |
| `model.json` | the model builds from that config; parameter counts per branch |
| `checkpoint.json` | missing / unexpected / shape-mismatched keys, mmcv cross-check |
| `batch.json` | input shapes (synthetic, config-shaped) and the reason |
| `forward.json` | full forward result, per-stage finiteness, depth tensor channels vs `dtransform` |
| `report.json` | all of the above plus pass/fail per acceptance criterion |

### If a cell fails

* **pinned wheels** — read `logs/pinned_wheels.log`; a missing cp38 wheel is a
  hard stop for the ticket (the pin cannot be built).
* **cuda toolchain** — `cuda_toolchain.json` records the three strategies tried.
  If it ended in `system-nvcc-with-guard-bypass`, the compile is still torch
  1.10.1+cu113, but nvcc is 12.x; the build log says whether the CUDA runtime ABI
  held.
* **build ops** — `logs/build_ops.log` is the full compiler output; the cell
  prints its tail. The failing extension name is in `build_ops.json`.
* **forward** — `logs/step_forward.log` plus `forward.json:forward_traceback`
  carry the exact stack; `forward.trunk_only_fallback` tells you whether only the
  head failed.

### What this ticket deliberately does not do

* No real nuScenes sample: infos do not exist yet (ticket 04 open item), so the
  forward uses a synthetic config-shaped batch and says so everywhere.
* No evaluation, no training, no spray transform — those are later tickets.
* No substitution of another BEVFusion implementation or a newer framework
  version: if the pinned stack cannot be built, the report says exactly where.
